# ACO Pheromone Distribution Animation

Reads PREACT drone telemetry from an `_output` folder and produces GIFs of the
pheromone fields (`tau_s` — staleness/freshness, `tau_v` — volume/density,
`tau_c` — crowding) on the road graph as the simulation progresses.

Each edge is colored by the pheromone value at that timestep; drones are
overlaid as scatter points so you can see which parts of the network the
swarm is currently exploring vs. reinforcing.

Expected inputs in `OUTPUT_DIR` (same `PREFIX`):
- `<prefix>_graph_edges.csv`
- `<prefix>_aco_edge_samples.csv`
- `<prefix>_state_samples.csv` (optional, used for drone overlay)
- `<prefix>_scan_events.csv` (optional)

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import animation
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm, Normalize

# ---- Configure paths ----
# Absolute path is safest — Jupyter's cwd depends on how the notebook was launched.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".git").exists()), None); assert ROOT is not None, "Run this notebook from within the WUInity repository."; OUTPUT_DIR = ROOT / "Examples/NFDRS4_Behave/Roxborough/_output"
PREFIX: str | None = None  # auto-detect latest *_aco_edge_samples.csv if None
GIF_FPS = 10
FRAME_STRIDE_SECONDS = 120.0  # render one frame every N simulation-seconds
MAX_FRAMES: int | None = None  # extra hard cap after striding; None = no cap

STATE_SUFFIX = "_state_samples.csv"
GRAPH_SUFFIX = "_graph_edges.csv"
EDGE_SUFFIX = "_aco_edge_samples.csv"
SCAN_SUFFIX = "_scan_events.csv"


def resolve_prefix(output_dir: Path, prefix: str | None) -> str:
    if prefix:
        return prefix
    candidates = sorted(output_dir.glob(f"*{EDGE_SUFFIX}"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError(f"No *{EDGE_SUFFIX} files in {output_dir}")
    return candidates[-1].name[: -len(EDGE_SUFFIX)]


if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"OUTPUT_DIR does not exist: {OUTPUT_DIR}")

prefix = resolve_prefix(OUTPUT_DIR, PREFIX)
print(f"Output dir: {OUTPUT_DIR}")
print(f"Prefix:     {prefix}")

graph_path = OUTPUT_DIR / f"{prefix}{GRAPH_SUFFIX}"
edge_path = OUTPUT_DIR / f"{prefix}{EDGE_SUFFIX}"
state_path = OUTPUT_DIR / f"{prefix}{STATE_SUFFIX}"
scan_path = OUTPUT_DIR / f"{prefix}{SCAN_SUFFIX}"

gif_dir = OUTPUT_DIR / f"{prefix}_pheromone_gifs"
gif_dir.mkdir(parents=True, exist_ok=True)
print(f"GIFs ->     {gif_dir}")


In [ ]:
# The edge-sample CSV can be huge (1+ GB). Load only the columns we need,
# then immediately downsample to a target timestep before pivoting.
EDGE_COLS = ["sim_time_s","edge_index","tau_s","tau_v","tau_c","occupancy_count"]
edge_df = pd.read_csv(edge_path, usecols=EDGE_COLS)
graph_df = pd.read_csv(graph_path)


def safe_read_csv(path):
    """Return a DataFrame or None — tolerate missing, empty, or header-only files."""
    if not path.exists() or path.stat().st_size == 0:
        return None
    try:
        df = pd.read_csv(path)
        return df if len(df) > 0 else None
    except pd.errors.EmptyDataError:
        return None


state_df = safe_read_csv(state_path)
scan_df  = safe_read_csv(scan_path)
print(f"state samples: {0 if state_df is None else len(state_df)}")
print(f"scan events:   {0 if scan_df  is None else len(scan_df)}")

graph_df = graph_df.set_index("edge_index").sort_index()
segments = np.stack(
    [
        graph_df[["start_x", "start_y"]].to_numpy(),
        graph_df[["end_x", "end_y"]].to_numpy(),
    ],
    axis=1,
)
edge_indices = graph_df.index.to_numpy()

# Pick frame times by bucketing sim_time into FRAME_STRIDE_SECONDS bins and
# taking the nearest available sample in each bucket. This keeps the pivot
# tractable even when the raw CSV samples at 1 Hz over thousands of seconds.
all_times = np.sort(edge_df["sim_time_s"].unique())
t_min, t_max = float(all_times.min()), float(all_times.max())
bucket = np.round((all_times - t_min) / FRAME_STRIDE_SECONDS).astype(np.int64)
_first_in_bucket = np.concatenate(([True], np.diff(bucket) > 0))
times = all_times[_first_in_bucket]
if MAX_FRAMES is not None and len(times) > MAX_FRAMES:
    step = max(1, len(times) // MAX_FRAMES)
    times = times[::step]

# Restrict the dataframe to only the frame timestamps before pivoting — this
# is the big speed win vs. pivoting the full 1 Hz dump.
edge_df = edge_df[edge_df["sim_time_s"].isin(times)].copy()

print(f"Edges: {len(edge_indices)}, raw timesteps: {len(all_times)}, frames: {len(times)}")
print(f"Time range: {t_min:.1f}s .. {t_max:.1f}s  (stride={FRAME_STRIDE_SECONDS}s)")
edge_df.head()


In [ ]:
# Pre-build per-frame value arrays for fast animation: shape (n_frames, n_edges).
def build_value_grid(edge_df: pd.DataFrame, field: str) -> np.ndarray:
    pivot = (
        edge_df.pivot_table(index="sim_time_s", columns="edge_index", values=field, aggfunc="last")
        .reindex(index=times, columns=edge_indices)
    )
    pivot = pivot.ffill().fillna(0.0)
    return pivot.to_numpy()


values = {f: build_value_grid(edge_df, f) for f in ("tau_s", "tau_v", "tau_c", "occupancy_count")}
for f, arr in values.items():
    print(f"{f:18s} min={arr.min():.4g}  max={arr.max():.4g}  mean={arr.mean():.4g}")

In [ ]:
# Bin drone positions by frame so we can overlay them.
# Telemetry is sampled every TelemetryIntervalSeconds (1s by default) while frames
# are FRAME_STRIDE_SECONDS apart (10s), so each frame bucket contains ~10 samples
# per drone. Plotting them all created a "trail" smear; instead we keep only the
# LATEST sample per (frame, drone) so each frame draws exactly one dot per drone.
drone_positions_per_frame: list[np.ndarray] = []
if state_df is not None and len(state_df) > 0 and "drone_id" in state_df.columns:
    state_times = state_df["sim_time_s"].to_numpy()
    frame_idx = np.searchsorted(times, state_times, side="right") - 1
    frame_idx = np.clip(frame_idx, 0, len(times) - 1)
    binned = state_df.assign(_frame=frame_idx)
    # Sort so the last row per (frame, drone) is the latest sample in that bin,
    # then drop_duplicates keeping the last → one position per drone per frame.
    binned = (
        binned.sort_values(["_frame", "drone_id", "sim_time_s"])
              .drop_duplicates(subset=["_frame", "drone_id"], keep="last")
    )
    grouped = binned.groupby("_frame")[["x", "y"]]
    for i in range(len(times)):
        if i in grouped.groups:
            drone_positions_per_frame.append(grouped.get_group(i).to_numpy())
        else:
            drone_positions_per_frame.append(np.empty((0, 2)))
else:
    drone_positions_per_frame = [np.empty((0, 2))] * len(times)
nonempty = [len(p) for p in drone_positions_per_frame if len(p)]
print(f"Drone overlay frames: {len(nonempty)} / {len(times)}")
if nonempty:
    print(f"Drones per frame (median): {int(np.median(nonempty))}")
else:
    print("Drones per frame (median): n/a — no drone state telemetry in this run.")


In [ ]:
def make_animation(
    field: str,
    grid: np.ndarray,
    *,
    cmap: str = "viridis",
    log: bool = True,
    title_prefix: str | None = None,
    vmax_percentile: float = 99.0,
    vmin_floor: float = 1e-3,
) -> animation.FuncAnimation:
    """Render an animation of `grid` (shape n_frames × n_edges) over the road network.

    The vmax is taken at `vmax_percentile` across the whole grid rather than the
    raw maximum — this prevents a single momentary spike (e.g. 12 drones piling
    on one edge, pushing tau_s to 120 for one frame) from collapsing the rest of
    the colour scale to zero. Use `vmax_percentile=100` for the raw max.
    """
    finite = grid[np.isfinite(grid)]
    if finite.size == 0 or float(np.nanmax(grid)) <= 0:
        vmax = 1.0
    else:
        vmax = max(float(np.percentile(finite, vmax_percentile)), vmin_floor * 10)
    if log:
        pos = grid[grid > 0]
        vmin = max(float(pos.min()) if pos.size else vmin_floor, vmin_floor)
        # If the percentile clipped under vmin, push it up to avoid LogNorm errors.
        if vmax <= vmin:
            vmax = vmin * 10
        norm = LogNorm(vmin=vmin, vmax=vmax)
    else:
        norm = Normalize(vmin=0.0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(9, 9))
    lc = LineCollection(segments, cmap=cmap, norm=norm, linewidths=1.4)
    lc.set_array(grid[0])
    ax.add_collection(lc)

    drone_scat = ax.scatter([], [], s=18, edgecolors="white", facecolors="red", linewidths=0.5, zorder=5)

    pad_x = 0.02 * (segments[..., 0].max() - segments[..., 0].min() + 1e-6)
    pad_y = 0.02 * (segments[..., 1].max() - segments[..., 1].min() + 1e-6)
    ax.set_xlim(segments[..., 0].min() - pad_x, segments[..., 0].max() + pad_x)
    ax.set_ylim(segments[..., 1].min() - pad_y, segments[..., 1].max() + pad_y)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(True, alpha=0.15)
    cbar = fig.colorbar(lc, ax=ax, shrink=0.8)
    cbar.set_label(f"{field}  (vmax = p{vmax_percentile:g} = {vmax:.3g}, raw_max = {float(np.nanmax(grid)):.3g})")

    title = ax.set_title("")

    def update(frame: int):
        lc.set_array(grid[frame])
        pos = drone_positions_per_frame[frame]
        drone_scat.set_offsets(pos if len(pos) else np.empty((0, 2)))
        label = f"{title_prefix or field}  t = {times[frame]:0.1f}s  ({frame + 1}/{len(times)})"
        title.set_text(label)
        return lc, drone_scat, title

    anim = animation.FuncAnimation(
        fig, update, frames=len(times), interval=1000 / GIF_FPS, blit=False
    )
    return anim, fig


In [ ]:
# Render tau_s — drops with time-since-scan, so high values = stale roads.
anim, fig = make_animation("tau_s", values["tau_s"], cmap="magma", title_prefix="τ_s (staleness)")
out = gif_dir / "tau_s.gif"
anim.save(out, writer=animation.PillowWriter(fps=GIF_FPS))
plt.close(fig)
print(out)

In [ ]:
# Render tau_v — vehicle-density evidence deposited by scans.
anim, fig = make_animation("tau_v", values["tau_v"], cmap="viridis", title_prefix="τ_v (vehicle evidence)")
out = gif_dir / "tau_v.gif"
anim.save(out, writer=animation.PillowWriter(fps=GIF_FPS))
plt.close(fig)
print(out)


In [ ]:
# Render tau_c — anti-crowding pheromone (drones already on edge).
anim, fig = make_animation("tau_c", values["tau_c"], cmap="plasma", title_prefix="τ_c (crowding)")
out = gif_dir / "tau_c.gif"
anim.save(out, writer=animation.PillowWriter(fps=GIF_FPS))
plt.close(fig)
print(out)

In [ ]:
# Render raw occupancy count for sanity-checking against tau_c.
anim, fig = make_animation(
    "occupancy_count", values["occupancy_count"], cmap="cividis", title_prefix="occupancy"
)
out = gif_dir / "occupancy.gif"
anim.save(out, writer=animation.PillowWriter(fps=GIF_FPS))
plt.close(fig)
print(out)

## Aggregate sanity plots

Quick non-animated context so you can correlate the GIFs with the global trend.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(times, values["tau_s"].mean(axis=1), label="τ_s mean")
axes[0].plot(times, values["tau_v"].mean(axis=1), label="τ_v mean")
axes[0].plot(times, values["tau_c"].mean(axis=1), label="τ_c mean")
axes[0].set_ylabel("mean pheromone")
axes[0].legend(); axes[0].grid(alpha=0.2)
axes[1].plot(times, values["occupancy_count"].sum(axis=1), color="black")
axes[1].set_ylabel("Σ occupancy")
axes[1].set_xlabel("sim time [s]")
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(gif_dir / "pheromone_global_stats.png", dpi=150)
fig

## New diagnostics: γ_e, d_ref, and the time-integrated scan density

After fixing the two implementation bugs (`ComputeMeasuredDensityPerLane` ignoring the time integral, and γ_e never being loaded from the SUMO XML), the telemetry now contains:

- `_graph_edges.csv` adds **`d_bar_e`** (per-edge mean per-lane density from the reference SUMO run), **`gamma`** (`d_bar_e / d_ref` per Definition 2), **`d_ref`** (network reference density, same on every row), **`t_ref_seconds`** (reference-run duration the loader observed in the XML).
- `_scan_events.csv` adds **`scan_seconds`** (`T_e`, the integration window), **`accumulated_vehicle_seconds`** (`∫ n_e dt`), **`normalised_density`** (`d̃_e^(scan) = d̃_scan / d_ref` per eq. dtilde), **`tau_v_deposit`** (`Q_v · d̃_e^(scan)`, the value that actually goes into τ_v per eq. 6), and **`d_ref`** copied alongside for self-contained rows.

The cells below visualise the new columns. They expect a re-run with the patched build.


In [ ]:
# Re-read the graph CSV in case the patched run added new columns.
graph_full = pd.read_csv(graph_path)
have_gamma = "gamma" in graph_full.columns
have_dbar = "d_bar_e" in graph_full.columns
have_dref = "d_ref" in graph_full.columns
d_ref_val = float(graph_full["d_ref"].iloc[0]) if have_dref and len(graph_full) else float("nan")
t_ref_val = float(graph_full["t_ref_seconds"].iloc[0]) if "t_ref_seconds" in graph_full.columns and len(graph_full) else float("nan")
print(f"graph columns: {list(graph_full.columns)}")
print(f"d_ref (veh/m/lane) = {d_ref_val}")
print(f"t_ref (s)         = {t_ref_val}")

if have_gamma and have_dbar:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    g_pos = graph_full[graph_full["gamma"] > 0]["gamma"]
    axes[0].hist(g_pos, bins=60, log=True, color="steelblue", edgecolor="white")
    axes[0].axvline(1.0, color="red", linestyle="--", label="γ=1 (network mean)")
    axes[0].set_xlabel("γ_e = d̄_e / d_ref")
    axes[0].set_ylabel("# edges (log)")
    axes[0].set_title(f"γ_e distribution  (n_pos={len(g_pos)}, n_zero={(graph_full['gamma']==0).sum()})")
    axes[0].legend(); axes[0].grid(alpha=0.2)

    # spatial map of gamma on the road graph
    segs_full = np.stack([
        graph_full[["start_x","start_y"]].to_numpy(),
        graph_full[["end_x","end_y"]].to_numpy(),
    ], axis=1)
    from matplotlib.collections import LineCollection as _LC
    from matplotlib.colors import LogNorm as _LN
    gvals = graph_full["gamma"].to_numpy().astype(float)
    gvals_safe = np.where(gvals > 0, gvals, np.nan)
    lc = _LC(segs_full, cmap="coolwarm", norm=_LN(vmin=max(1e-3, np.nanpercentile(gvals_safe, 5)),
                                                   vmax=max(1e-2, np.nanpercentile(gvals_safe, 99))),
             linewidths=1.4)
    lc.set_array(gvals_safe)
    axes[1].add_collection(lc)
    axes[1].set_xlim(segs_full[...,0].min(), segs_full[...,0].max())
    axes[1].set_ylim(segs_full[...,1].min(), segs_full[...,1].max())
    axes[1].set_aspect("equal"); axes[1].grid(alpha=0.15)
    axes[1].set_title("γ_e on the network (red = important, blue = quiet)")
    fig.colorbar(lc, ax=axes[1], label="γ_e")
    fig.tight_layout()
    fig.savefig(gif_dir / "gamma_distribution.png", dpi=150)
    plt.show()
else:
    print("[!] gamma / d_bar_e not present — the run was made with the pre-patch build.")


In [ ]:
# Scan-event density: with the eq. (10) fix, the time-integrated value should be
# nonzero on far more scans than the previous instantaneous-snapshot version.
if scan_df is not None and len(scan_df):
    sd = scan_df.copy()
    has_int = "accumulated_vehicle_seconds" in sd.columns and "scan_seconds" in sd.columns
    print(f"scan rows: {len(sd)}, columns: {list(sd.columns)}")
    print(f"measured_density_per_lane: frac_zero={(sd['measured_density_per_lane']==0).mean():.3f}, mean={sd['measured_density_per_lane'].mean():.4g}, max={sd['measured_density_per_lane'].max():.4g}")

    fig, axes = plt.subplots(1, 2 if has_int else 1, figsize=(13, 4), squeeze=False)
    ax = axes[0, 0]
    pos = sd[sd["measured_density_per_lane"] > 0]["measured_density_per_lane"]
    if len(pos):
        ax.hist(pos, bins=60, log=True, color="seagreen", edgecolor="white")
    ax.set_xlabel("measured_density_per_lane (veh/m/lane)")
    ax.set_ylabel("# scans (log)")
    ax.set_title(f"Scan density distribution (nonzero scans: {len(pos)}/{len(sd)})")
    ax.grid(alpha=0.2)

    if has_int:
        ax2 = axes[0, 1]
        # Vehicle-seconds per scan: how many "vehicle-seconds" were captured during T_e
        ax2.scatter(sd["scan_seconds"], sd["accumulated_vehicle_seconds"], s=4, alpha=0.4)
        ax2.set_xlabel("scan_seconds  T_e")
        ax2.set_ylabel("∫ n_e(t) dt   [vehicle·s]")
        ax2.set_title("Scan integration window vs. vehicles captured")
        ax2.grid(alpha=0.2)

    fig.tight_layout()
    fig.savefig(gif_dir / "scan_density_distribution.png", dpi=150)
    plt.show()

    if "tau_v_deposit" in sd.columns:
        print(f"\ntau_v_deposit: nonzero={ (sd['tau_v_deposit']>0).sum() }/{ len(sd) }, "
              f"mean={sd['tau_v_deposit'].mean():.4g}, max={sd['tau_v_deposit'].max():.4g}")
        print(f"normalised_density (d̃): nonzero={ (sd['normalised_density']>0).sum() }/{ len(sd) }, "
              f"mean={sd['normalised_density'].mean():.4g}, max={sd['normalised_density'].max():.4g}")
else:
    print("no scan events file")


## Validation: swarm-observed vs SUMO-true vehicle count

This is the operational accuracy metric. At each simulation time `t`:

- **Ground truth** = sum of `sampledSeconds` over all edges in the 1-second interval
  containing `t` (from the live SUMO meandata XML the current run produced). For a
  1-second sampling interval, `sampledSeconds` ≈ instantaneous vehicle count.
- **Swarm estimate** = for each edge, take the most recent completed scan whose
  age is ≤ `FRESHNESS_WINDOW_S`. Sum `estimated_vehicle_count` across all edges
  with a fresh-enough scan. Edges never scanned, or last scanned too long ago,
  contribute zero — the swarm legitimately doesn't know about them.

If the orange line tracks the blue line, the swarm is maintaining a representative
picture of network density. Persistent under-tracking ⇒ coverage is too sparse or
freshness window is too short; over-tracking ⇒ stale scans inflating the count.

In [ ]:
import xml.etree.ElementTree as ET

# How long a scan's reading is allowed to "count" toward the swarm's current estimate.
# Tune this to test different operational latencies; 120s is a reasonable starting point.
FRESHNESS_WINDOW_S = 120.0
# Sample the comparison at this cadence (seconds). Finer is slower but more detail.
SAMPLE_EVERY_S = 10.0

LIVE_DUMP = OUTPUT_DIR / "edge_density_1s.xml"
TRUTH_CACHE = OUTPUT_DIR / f"{prefix}_ground_truth_total.csv"


def load_or_parse_truth(dump_path: Path, cache_path: Path) -> pd.DataFrame:
    """Per-second total vehicles in the network, summed across all edges."""
    if cache_path.exists() and cache_path.stat().st_mtime >= dump_path.stat().st_mtime:
        print(f"loaded cached ground truth: {cache_path.name}")
        return pd.read_csv(cache_path)
    if not dump_path.exists():
        raise FileNotFoundError(f"SUMO meandata not found: {dump_path}")
    print(f"streaming {dump_path.name} ({dump_path.stat().st_size / 1e9:.2f} GB) — first time, will cache...")
    rows = []
    cur_t, cur_total = None, 0.0
    for _, elem in ET.iterparse(str(dump_path), events=("end",)):
        if elem.tag == "interval":
            if cur_t is not None:
                rows.append((cur_t, cur_total))
            try:
                cur_t = float(elem.get("begin", "nan"))
            except (TypeError, ValueError):
                cur_t = None
            cur_total = 0.0
            elem.clear()
        elif elem.tag == "edge":
            try:
                cur_total += float(elem.get("sampledSeconds", "0"))
            except (TypeError, ValueError):
                pass
            elem.clear()
    if cur_t is not None:
        rows.append((cur_t, cur_total))
    df = pd.DataFrame(rows, columns=["sim_time_s", "ground_truth_vehicles"])
    df.to_csv(cache_path, index=False)
    print(f"cached -> {cache_path.name}")
    return df


truth_df = load_or_parse_truth(LIVE_DUMP, TRUTH_CACHE)
print(f"truth rows: {len(truth_df)},  time=[{truth_df.sim_time_s.min()}, {truth_df.sim_time_s.max()}]")
print(f"truth peak vehicles: {truth_df.ground_truth_vehicles.max():.1f}")

# Build per-edge scan timeline: arrays of (scan_time, estimated_vehicle_count) per edge.
edge_scan_arrays = {}
if scan_df is not None and len(scan_df):
    for ei, g in scan_df.sort_values("sim_time_s").groupby("edge_index"):
        edge_scan_arrays[int(ei)] = (
            g["sim_time_s"].to_numpy(),
            g["estimated_vehicle_count"].to_numpy(),
        )

t_min = float(truth_df.sim_time_s.min())
t_max = float(truth_df.sim_time_s.max())
query_t = np.arange(t_min, t_max + 1e-6, SAMPLE_EVERY_S)
swarm_estimate = np.zeros_like(query_t)
edges_in_view = np.zeros_like(query_t, dtype=np.int64)

for ei, (st, sc) in edge_scan_arrays.items():
    # For each query time, find the latest scan whose time <= query_t.
    idx = np.searchsorted(st, query_t, side="right") - 1
    valid = idx >= 0
    safe_idx = np.where(valid, idx, 0)
    last_scan_t = st[safe_idx]
    last_scan_count = sc[safe_idx]
    fresh = valid & ((query_t - last_scan_t) <= FRESHNESS_WINDOW_S)
    swarm_estimate += np.where(fresh, last_scan_count, 0.0)
    edges_in_view += fresh.astype(np.int64)

# Resample truth onto the query grid (nearest 1-second bin).
truth_lookup = dict(zip(truth_df.sim_time_s.astype(int), truth_df.ground_truth_vehicles))
truth_total = np.array([truth_lookup.get(int(t), 0.0) for t in query_t])

n_edges_total = len(graph_full) if "graph_full" in dir() else len(pd.read_csv(graph_path))

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(query_t, truth_total, color="steelblue", linewidth=1.3, label="SUMO ground truth")
axes[0].plot(query_t, swarm_estimate, color="darkorange", linewidth=1.1,
             label=f"swarm estimate (window={FRESHNESS_WINDOW_S:.0f}s)")
axes[0].set_ylabel("# vehicles in network")
axes[0].set_title("Validation: total vehicles — observed vs ground truth")
axes[0].legend(loc="upper left"); axes[0].grid(alpha=0.25)

ratio = np.divide(swarm_estimate, truth_total, out=np.full_like(swarm_estimate, np.nan, dtype=float), where=truth_total > 1.0)
axes[1].axhline(1.0, color="grey", linewidth=0.8, linestyle="--", label="perfect")
axes[1].plot(query_t, ratio, color="green", linewidth=1.0)
axes[1].set_ylabel("estimate / truth")
axes[1].set_ylim(0, max(2.0, np.nanpercentile(ratio[~np.isnan(ratio)], 99) if np.any(~np.isnan(ratio)) else 2.0))
axes[1].grid(alpha=0.25); axes[1].legend(loc="upper left")


# "Tracked fraction" metrics (separate from MAPE):
# - ratio: raw estimate/truth (can exceed 1 if overestimating)
# - tracked_fraction: clipped to [0,1], interpreted as "fraction of true vehicles represented"
tracked_fraction = np.where(np.isnan(ratio), np.nan, np.clip(ratio, 0.0, 1.0))
untracked_fraction = np.where(np.isnan(tracked_fraction), np.nan, 1.0 - tracked_fraction)

coverage_pct = 100.0 * edges_in_view / max(1, n_edges_total)
axes[2].plot(query_t, coverage_pct, color="purple", linewidth=1.0)
axes[2].set_ylabel("% edges with a fresh scan")
axes[2].set_xlabel("simulation time [s]")
axes[2].grid(alpha=0.25)

fig.tight_layout()
fig.savefig(gif_dir / "validation_truth_vs_swarm.png", dpi=150)
plt.show()

# Headline numbers.
mask = truth_total > 1.0
mae = np.mean(np.abs(swarm_estimate[mask] - truth_total[mask]))
mape = np.mean(np.abs(swarm_estimate[mask] - truth_total[mask]) / truth_total[mask]) * 100

mean_ratio_pct = np.nanmean(ratio) * 100.0
median_ratio_pct = np.nanmedian(ratio) * 100.0
mean_tracked_pct = np.nanmean(tracked_fraction) * 100.0
mean_untracked_pct = np.nanmean(untracked_fraction) * 100.0

print(f"\n=== Validation summary (FRESHNESS_WINDOW={FRESHNESS_WINDOW_S:.0f}s) ===")
print(f"  MAE (vehicles):         {mae:.1f}")
print(f"  MAPE (% of truth):      {mape:.1f}%")
print(f"  mean estimate/truth:    {mean_ratio_pct:.1f}%")
print(f"  median estimate/truth:  {median_ratio_pct:.1f}%")
print(f"  mean tracked fraction (clipped 0-100%):   {mean_tracked_pct:.1f}%")
print(f"  mean untracked fraction (clipped 0-100%): {mean_untracked_pct:.1f}%")
print(f"  mean coverage (% edges with fresh scan): {coverage_pct.mean():.1f}%")
print(f"  peak coverage:          {coverage_pct.max():.1f}%")


## VDD analysis: shaded detection-deficit area

Same analysis as in `dronecount_sweep_compare.ipynb` but for *this* single run: the
**Vehicle Detection Curve** $\widehat{G}(t)$ (the swarm estimate computed above)
plotted against the **ground-truth curve** $G(t)$, with the region between them
shaded as the **detection deficit** $\Delta(t) = G(t) - \widehat{G}(t)$. Its area
is the **Vehicle Detection Deficit (VDD)**,
$$\mathrm{VDD} = \int_0^T \!\big(G(t) - \widehat{G}(t)\big)\,dt \quad [\,\text{veh·s}\,],$$
printed on the panel in both absolute (veh·s) and normalised (% of the total
surveillance demand $\int_0^T G\,dt$) form.

Two standalone PNGs are saved into the run's gif folder so they can be dropped
straight into a paper figure:

- `validation_vdc_vdd.png` — VDC vs $G(t)$ with shaded $\Delta(t)$ and the VDD annotation.
- `validation_coverage.png` — the per-second coverage curve (fraction of fresh edges).

In [ ]:
# Shaded-VDD figure for this single run. Reuses the arrays computed above
# (query_t, truth_total, swarm_estimate, coverage_pct) so no recomputation is needed.
# Matches the styling added to dronecount_sweep_compare.ipynb so figures from both
# notebooks look the same in the paper.

_trapz = getattr(np, "trapezoid", np.trapz)

# VDD = integral(G) - integral(G_hat) = area between the two curves [vehicle-seconds].
demand_int = float(_trapz(truth_total, query_t))
est_int    = float(_trapz(swarm_estimate, query_t))
vdd        = demand_int - est_int
vdd_pct    = 100.0 * vdd / max(1.0, demand_int)
n_drones   = state_df['drone_index'].nunique() if ('state_df' in dir() and state_df is not None and 'drone_index' in state_df.columns) else None

# --- Figure 1: VDC vs ground-truth curve with shaded detection deficit ---------
fig, ax = plt.subplots(figsize=(9, 5))
color = "tab:blue"
ax.plot(query_t, truth_total, color="black", linewidth=2.0,
        label="ground-truth curve $G(t)$", zorder=10)
n_label = f" — $N={n_drones}$ drones" if n_drones else ""
ax.plot(query_t, swarm_estimate, color=color, linewidth=1.8,
        label=f"VDC $\\widehat{{G}}(t)${n_label}", zorder=9)
ax.fill_between(query_t, swarm_estimate, truth_total,
                color=color, alpha=0.25, linewidth=0,
                label="detection deficit $\\Delta(t)$")

ax.text(0.03, 0.97,
        f"VDD = {vdd:,.0f} veh·s\n      = {vdd/1e3:,.1f} k veh·s\n      = {vdd_pct:.1f}% of demand",
        transform=ax.transAxes, ha="left", va="top",
        fontsize=11, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor=color, alpha=0.9))

ax.set_xlabel("simulation time [s]")
ax.set_ylabel("vehicles")
title = "ACO stigmergic policy — VDC vs ground-truth curve"
if n_drones:
    title += f" ($N={n_drones}$ drones)"
ax.set_title(title)
ax.grid(alpha=0.25)
ax.legend(loc="upper right", fontsize=9)
fig.tight_layout()
out_vdc = gif_dir / "validation_vdc_vdd.png"
fig.savefig(out_vdc, dpi=150, bbox_inches="tight")
print(f"saved: {out_vdc}")
plt.show()

# --- Figure 2: coverage curve (standalone) -------------------------------------
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(query_t, coverage_pct, color="tab:purple", linewidth=1.2,
        label=f"mean = {coverage_pct.mean():.1f}%, peak = {coverage_pct.max():.1f}%")
ax.set_xlabel("simulation time [s]")
ax.set_ylabel("fresh edges (% of $\\mathcal{U}$)")
ax.set_title("Coverage: fraction of edges with a fresh scan")
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
out_cov = gif_dir / "validation_coverage.png"
fig.savefig(out_cov, dpi=150, bbox_inches="tight")
print(f"saved: {out_cov}")
plt.show()

print(f"\n=== VDD summary ===")
print(f"  demand integral ∫G dt        = {demand_int:>12,.0f} veh·s")
print(f"  detected integral ∫Ĝ dt      = {est_int:>12,.0f} veh·s")
print(f"  VDD = ∫G dt - ∫Ĝ dt          = {vdd:>12,.0f} veh·s  ({vdd/1e3:.1f} k veh·s)")
print(f"  VDD as % of demand          = {vdd_pct:>12.2f}%")
print(f"  fraction of vehicle-seconds detected = {100.0 - vdd_pct:.2f}%")

## Hyperparameter proportion diagnostics

Four panels to make the algorithm's internals visible:

1. **Pheromone value distributions** — are τ_s, τ_v, τ_c in single-digit / dozens / runaway-thousands territory? Tells you whether `Q*` and `Rho*` are balanced.
2. **Decision-rule factor distributions** — for each (edge, time) sample, compute the three multiplicative factors `(1+τ_s)^(-α)`, `(1+τ_v)^β`, `(1+τ_c)^(-δ)`. If one factor has 10⁶× the spread of the others, it's the only thing steering decisions — the others are dead knobs.
3. **Effective half-lives by γ tier** — γ-amplified decay rates per edge bucket. Visualises whether busy edges actually "forget" faster than quiet ones the way you intended.
4. **Inter-scan interval by γ tier** — how often does the swarm actually revisit busy vs quiet edges? Ground-truth check that the γ + decay machinery is producing the right behaviour.

The hyperparameters are read from the constants at the top — keep them in sync with the `.wui` you ran.

In [ ]:
# --- Hyperparameters auto-loaded from .wui [AcoStigmergicPolicy] ---
from pathlib import Path

# Optional override: set WUI_PATH manually if auto-detect fails.
WUI_PATH = None

def parse_wui_section(path: Path, section: str):
    kv = {}
    current = None
    with path.open() as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith('[') and line.endswith(']'):
                current = line[1:-1].strip()
                continue
            # Remove inline comments using ';' or '#'.
            for c in (';', '#'):
                if c in line:
                    line = line.split(c, 1)[0].strip()
            if not line or '=' not in line:
                continue
            if current == section:
                k, v = line.split('=', 1)
                kv[k.strip()] = v.strip()
    return kv

def resolve_wui_path():
    if WUI_PATH is not None:
        p = Path(WUI_PATH).expanduser().resolve()
        if p.exists():
            return p
        raise FileNotFoundError(f'WUI_PATH does not exist: {p}')

    # Common repo-relative location from this notebook.
    candidates = [
        Path.cwd() / '../../../../../Examples/NFDRS4_Behave/Roxborough/Roxborough_global_smoke_drones_aco.wui',
        Path.cwd() / '../../../../../../Examples/NFDRS4_Behave/Roxborough/Roxborough_global_smoke_drones_aco.wui',
        ROOT / "Examples/NFDRS4_Behave/Roxborough/Roxborough_global_smoke_drones_aco.wui",
    ]
    for c in candidates:
        c = c.resolve()
        if c.exists():
            return c

    raise FileNotFoundError('Could not auto-locate Roxborough_global_smoke_drones_aco.wui. Set WUI_PATH explicitly.')

def get_float(d, key, default):
    try:
        return float(d.get(key, default))
    except Exception:
        return float(default)

wui_path = resolve_wui_path()
aco_cfg = parse_wui_section(wui_path, 'AcoStigmergicPolicy')

# Defaults mirror notebook fallback, but normally values come from .wui.
ALPHA = get_float(aco_cfg, 'Alpha', 1.5)
BETA  = get_float(aco_cfg, 'Beta', 1.5)
DELTA = get_float(aco_cfg, 'Delta', 2.0)
RHO_S = get_float(aco_cfg, 'RhoS', 0.05)
RHO_V = get_float(aco_cfg, 'RhoV', 0.00025)
RHO_C = get_float(aco_cfg, 'RhoC', 1.0)
QS    = get_float(aco_cfg, 'Qs', 3.0)
QV    = get_float(aco_cfg, 'Qv', 2.0)
QC    = get_float(aco_cfg, 'Qc', 1.0)

print(f'Loaded ACO hyperparameters from: {wui_path}')
print(f'  Alpha={ALPHA}, Beta={BETA}, Delta={DELTA}')
print(f'  RhoS={RHO_S}, RhoV={RHO_V}, RhoC={RHO_C}')
print(f'  Qs={QS}, Qv={QV}, Qc={QC}')

# Subsample edge_df for speed; keeps the picture statistically faithful but plottable.
SAMPLE_MAX_ROWS = 500_000
if len(edge_df) > SAMPLE_MAX_ROWS:
    edge_sample = edge_df.sample(SAMPLE_MAX_ROWS, random_state=0)
else:
    edge_sample = edge_df

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ===== Panel 1: pheromone value distributions =====
ax = axes[0, 0]
for fld, colour in [("tau_s", "tab:blue"), ("tau_v", "tab:orange"), ("tau_c", "tab:green")]:
    pos = edge_sample[fld][edge_sample[fld] > 0]
    if len(pos) == 0:
        continue
    bins = np.logspace(np.log10(max(pos.min(), 1e-6)), np.log10(pos.max() + 1e-9), 60)
    ax.hist(pos, bins=bins, histtype="step", linewidth=1.6, color=colour,
            label=f"{fld}  (median={pos.median():.3g}, p99={pos.quantile(0.99):.3g})")
ax.set_xscale("log")
ax.set_xlabel("pheromone value (log)")
ax.set_ylabel("# (edge, time) samples")
ax.set_title("Pheromone value distributions (nonzero only)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(alpha=0.2, which="both")

# ===== Panel 2: decision-rule factor distributions =====
# For every sampled (edge, time), compute the three factors that go into P(e).
# If one factor's range dominates the others, that knob is doing all the steering.
ax = axes[0, 1]
f_s = np.power(1.0 + edge_sample["tau_s"].to_numpy(), -ALPHA)
f_v = np.power(1.0 + edge_sample["tau_v"].to_numpy(),  BETA)
f_c = np.power(1.0 + edge_sample["tau_c"].to_numpy(), -DELTA)
bp_data = [f_s, f_v, f_c]
bp_labels = [f"(1+τ_s)^-{ALPHA}", f"(1+τ_v)^{BETA}", f"(1+τ_c)^-{DELTA}"]
bp_colours = ["tab:blue", "tab:orange", "tab:green"]
parts = ax.boxplot(
    [np.log10(np.clip(d, 1e-30, None)) for d in bp_data],
    tick_labels=bp_labels, patch_artist=True, showfliers=False, widths=0.55,
)
for patch, c in zip(parts["boxes"], bp_colours):
    patch.set_facecolor(c); patch.set_alpha(0.45)
ax.axhline(0.0, color="grey", linewidth=0.7, linestyle="--", label="factor = 1 (neutral)")
ax.set_ylabel("log10(factor)")
ax.set_title("Decision-rule factor distributions across all samples")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.2)
# Numeric summary
for d, name in zip(bp_data, bp_labels):
    lo, med, hi = np.quantile(d, [0.01, 0.5, 0.99])
    spread_decades = np.log10(max(hi, 1e-30)) - np.log10(max(lo, 1e-30))
    print(f"  {name:18s}  p1={lo:.3g}  median={med:.3g}  p99={hi:.3g}  spread={spread_decades:.1f} decades")

# ===== Panel 3: effective decay half-lives by γ tier =====
# t_half(tau_s) = ln(2) / (RhoS * gamma); same for tau_v with RhoV.
ax = axes[1, 0]
if "gamma" in graph_full.columns:
    gammas = graph_full["gamma"].to_numpy().astype(float)
    gammas = np.clip(gammas, 1e-6, None)  # match TrySetGammaForEdge clamp
    # tier by quintile of gamma — first quintile is essentially "silent", last is "highway"
    quintiles = np.quantile(gammas, [0, 0.2, 0.4, 0.6, 0.8, 1.0])
    tier_labels = []
    for i in range(5):
        lo, hi = quintiles[i], quintiles[i + 1]
        mid = np.sqrt(max(lo, 1e-12) * max(hi, 1e-12))
        tier_labels.append(f"γ∈[{lo:.2g},{hi:.2g}]\n(geo-mean {mid:.2g})")
    g_mid = np.sqrt(quintiles[:-1].clip(1e-12) * quintiles[1:].clip(1e-12))
    t_half_s = np.log(2.0) / (RHO_S * g_mid)
    t_half_v = np.log(2.0) / (RHO_V * g_mid)
    x = np.arange(5)
    width = 0.38
    ax.bar(x - width/2, t_half_s, width, color="tab:blue", label="τ_s half-life")
    ax.bar(x + width/2, t_half_v, width, color="tab:orange", label="τ_v half-life")
    ax.set_yscale("log")
    ax.set_xticks(x)
    ax.set_xticklabels(tier_labels, fontsize=7)
    ax.set_ylabel("half-life [s] (log)")
    ax.set_title(f"Effective decay half-life by γ tier  (RhoS={RHO_S}, RhoV={RHO_V})")
    ax.axhline(60.0, color="grey", linestyle="--", linewidth=0.7, label="1 min")
    ax.axhline(1.0, color="grey", linestyle=":",  linewidth=0.7, label="1 s")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2, which="both")
else:
    ax.text(0.5, 0.5, "no gamma column in graph_edges.csv", ha="center", transform=ax.transAxes)

# ===== Panel 4: inter-scan interval per edge, by γ tier =====
ax = axes[1, 1]
if scan_df is not None and len(scan_df) > 1 and "gamma" in graph_full.columns:
    # Build a per-edge gamma lookup.
    gamma_by_edge = graph_full[["gamma"]].copy()
    gamma_by_edge.index.name = "edge_index"
    sd = scan_df.sort_values(["edge_index", "sim_time_s"]).copy()
    sd["scan_gap"] = sd.groupby("edge_index")["sim_time_s"].diff()
    sd = sd.dropna(subset=["scan_gap"])
    sd = sd.merge(gamma_by_edge, left_on="edge_index", right_index=True)
    sd["gamma_safe"] = np.clip(sd["gamma"].astype(float), 1e-6, None)
    # Bin by quintile of edge γ.
    sd["g_tier"] = pd.qcut(sd["gamma_safe"], q=5, labels=False, duplicates="drop")
    box_data = [sd[sd["g_tier"] == t]["scan_gap"].to_numpy() for t in sorted(sd["g_tier"].unique())]
    labels = []
    for t in sorted(sd["g_tier"].unique()):
        gs = sd[sd["g_tier"] == t]["gamma_safe"]
        labels.append(f"γ∈[{gs.min():.2g},{gs.max():.2g}]")
    ax.boxplot(box_data, tick_labels=labels, showfliers=False, widths=0.55)
    ax.set_yscale("log")
    ax.set_ylabel("inter-scan gap [s] (log)")
    ax.set_xlabel("γ tier (busy →)")
    ax.set_title(f"Per-edge revisit interval  ({len(sd)} gaps over {sd['edge_index'].nunique()} edges)")
    ax.grid(alpha=0.2, which="both")
    medians = [np.median(d) if len(d) else float("nan") for d in box_data]
    print(f"\nMedian revisit interval by γ tier (s): {[f'{m:.1f}' for m in medians]}")
else:
    ax.text(0.5, 0.5, "not enough scan events to compute revisit gaps", ha="center", transform=ax.transAxes)

fig.tight_layout()
fig.savefig(gif_dir / "hyperparameter_proportions.png", dpi=150)
plt.show()


## Textual diagnostics — pheromone proportions & effects

Heavy-text summary of what the swarm is actually doing. Eight sections:

1. **Time / coverage** — how much of the network the swarm explored, and how fast.
2. **Pheromone magnitude proportions** — what range each τ field actually reached, by γ tier.
3. **Decision-rule factor proportions** — which multiplicative term has the most swing across (edge, time) samples.
4. **Local decision dominance** — at scan-completion moments, which factor was driving the choice that brought the drone here?
5. **Scan locality** — for consecutive scans by the same drone, fraction same-edge / sibling-edge / further-hop. Reveals whether the swarm loops, walks, or jumps.
6. **τ_v informativeness** — does pre-scan τ_v actually predict the density the drone is about to measure? Tests whether the memory signal is useful.
7. **Edge concentration** — Gini-like measure of how lopsided the scan distribution is.
8. **Verdict** — a few heuristic flags ("β has no measurable effect", "swarm is in lock-in", etc.) based on the numbers above.

In [ ]:
# --- Hyperparameters used by the run (keep in sync with .wui) ---
# These reproduce the algorithmic effects below; if the .wui differs, change here too.
H_ALPHA = ALPHA if "ALPHA" in dir() else 1.5
H_BETA  = BETA  if "BETA"  in dir() else 1.5
H_DELTA = DELTA if "DELTA" in dir() else 2.0
H_RHO_S = RHO_S if "RHO_S" in dir() else 0.05
H_RHO_V = RHO_V if "RHO_V" in dir() else 0.00025
H_QS    = QS    if "QS"    in dir() else 3.0
H_QV    = QV    if "QV"    in dir() else 2.0
H_USE_REF_ONLINE = False  # set to True if your .wui has UseReferenceWeightsOnline=true

def banner(title):
    print()
    print("=" * 78)
    print(f"  {title}")
    print("=" * 78)


# ============================================================
# 1. TIME / COVERAGE
# ============================================================
banner("1. TIME / COVERAGE")
n_edges = len(graph_full)
t0 = float(min(edge_df["sim_time_s"].min(), scan_df["sim_time_s"].min() if scan_df is not None else float("inf")))
t1 = float(edge_df["sim_time_s"].max())
duration = t1 - t0
n_drones = scan_df["drone_id"].nunique() if scan_df is not None else 0
n_scans  = len(scan_df) if scan_df is not None else 0
edges_ever_scanned = scan_df["edge_index"].nunique() if scan_df is not None else 0
print(f"Run duration:             {duration:>8.1f} s   ({duration/60:.1f} min)")
print(f"Drones:                   {n_drones:>8d}")
print(f"Total scans:              {n_scans:>8d}")
print(f"Mean scans/drone/min:     {n_scans/max(1,n_drones)/max(1,duration/60):>8.2f}")
print(f"Edges in graph:           {n_edges:>8d}")
print(f"Edges ever scanned:       {edges_ever_scanned:>8d}   ({edges_ever_scanned/n_edges*100:.1f}%)")

# Coverage growth curve (how fast did the swarm reach K% coverage?).
if scan_df is not None and len(scan_df):
    first_scan_t = scan_df.sort_values("sim_time_s").drop_duplicates("edge_index", keep="first")
    first_scan_t = first_scan_t.sort_values("sim_time_s")
    cumul = np.arange(1, len(first_scan_t) + 1) / n_edges * 100.0
    targets = [10.0, 25.0, 50.0, 75.0, 90.0]
    for tgt in targets:
        ix = np.searchsorted(cumul, tgt)
        if ix < len(first_scan_t):
            t_reach = first_scan_t["sim_time_s"].iloc[ix]
            print(f"Time to reach {tgt:>4.0f}% coverage: {t_reach - t0:>7.1f} s")
        else:
            print(f"Time to reach {tgt:>4.0f}% coverage: NOT REACHED  (final={cumul[-1]:.1f}%)")

# ============================================================
# 2. PHEROMONE MAGNITUDE PROPORTIONS (by gamma tier)
# ============================================================
banner("2. PHEROMONE MAGNITUDES BY γ TIER")
# Join edge samples to per-edge gamma (one bucket per edge for the whole run).
gamma_lookup = graph_full[["gamma"]].astype(float)
gamma_lookup.index.name = "edge_index"
samp = edge_df.merge(gamma_lookup, left_on="edge_index", right_index=True, how="left")
samp["gamma_safe"] = np.clip(samp["gamma"].astype(float), 1e-6, None)
samp["g_tier"] = pd.qcut(samp["gamma_safe"], q=4, labels=["silent","quiet","busy","highway"], duplicates="drop")
print(f"{'tier':>9s} {'γ range':>22s} | {'τ_s p99':>10s} {'τ_v p99':>10s} {'τ_c p99':>10s} {'occ max':>8s}  {'frac(τ_v>0)':>11s}")
for tier in ["silent","quiet","busy","highway"]:
    g = samp[samp["g_tier"] == tier]
    if len(g) == 0: continue
    g_lo, g_hi = g["gamma_safe"].min(), g["gamma_safe"].max()
    p99_s = g["tau_s"].quantile(0.99)
    p99_v = g["tau_v"].quantile(0.99)
    p99_c = g["tau_c"].quantile(0.99)
    occ_max = int(g["occupancy_count"].max())
    f_v_pos = (g["tau_v"] > 0).mean() * 100
    print(f"{tier:>9s}  [{g_lo:>8.2g},{g_hi:>8.2g}] | {p99_s:>10.3g} {p99_v:>10.3g} {p99_c:>10.3g} {occ_max:>8d}  {f_v_pos:>10.2f}%")

# ============================================================
# 3. DECISION-RULE FACTOR PROPORTIONS (global)
# ============================================================
banner("3. DECISION-RULE FACTOR PROPORTIONS (all (edge,time) samples)")
sub = edge_df.sample(min(300_000, len(edge_df)), random_state=0) if len(edge_df) > 300_000 else edge_df
f_s = np.power(1.0 + sub["tau_s"].to_numpy(), -H_ALPHA)
f_v = np.power(1.0 + sub["tau_v"].to_numpy(),  H_BETA)
f_c = np.power(1.0 + sub["tau_c"].to_numpy(), -H_DELTA)
print(f"With Alpha={H_ALPHA}, Beta={H_BETA}, Delta={H_DELTA}:")
print(f"{'factor':>16s}  {'p1':>10s}  {'p50':>8s}  {'p99':>10s}  {'p99/p1':>10s}  {'spread (decades)':>18s}")
for name, arr in [(f"(1+τ_s)^-{H_ALPHA}", f_s), (f"(1+τ_v)^{H_BETA}", f_v), (f"(1+τ_c)^-{H_DELTA}", f_c)]:
    p1, p50, p99 = np.percentile(arr, [1, 50, 99])
    ratio = p99 / max(p1, 1e-30)
    spread = np.log10(max(p99, 1e-30)) - np.log10(max(p1, 1e-30))
    print(f"{name:>16s}  {p1:>10.3g}  {p50:>8.3g}  {p99:>10.3g}  {ratio:>10.3g}  {spread:>17.2f}")
print()
print("Reading: a factor with spread < 0.5 decades is effectively constant — that hyperparameter")
print("is not influencing decisions in this run. Spread > 5 decades means it is overwhelmingly")
print("dominant; the other terms can't compete.")

# ============================================================
# 4. LOCAL DECISION DOMINANCE (at scan moments)
# ============================================================
# When a scan completes on edge e at time t, the drone CHOSE e at the prior node a few
# seconds earlier. We reconstruct the pheromone state on e just before that scan
# (use the latest edge_df sample within 5s before t) and compute the three factors.
# Whichever factor is most extreme (largest log-deviation from 1) is the "dominant" one.
banner("4. LOCAL DECISION DOMINANCE (factor most-deviating from neutral at scan time)")
if scan_df is not None and len(scan_df):
    # Build a per-(edge,time) lookup as a multi-index Series for fast joins.
    snap = edge_df[["sim_time_s","edge_index","tau_s","tau_v","tau_c"]].copy()
    snap = snap.sort_values(["edge_index","sim_time_s"])
    sc = scan_df[["sim_time_s","edge_index"]].copy().sort_values(["edge_index","sim_time_s"])
    # asof-merge per edge: for each scan, find the latest pheromone sample <= scan time.
    merged_chunks = []
    for ei in sc["edge_index"].unique():
        sn = snap[snap["edge_index"] == ei]
        sc_ei = sc[sc["edge_index"] == ei]
        if len(sn) == 0 or len(sc_ei) == 0: continue
        m = pd.merge_asof(sc_ei.sort_values("sim_time_s"), sn.sort_values("sim_time_s"),
                          on="sim_time_s", direction="backward", suffixes=("_scan","_snap"))
        merged_chunks.append(m)
    if merged_chunks:
        scan_states = pd.concat(merged_chunks).dropna(subset=["tau_s","tau_v","tau_c"])
        fs = np.power(1.0 + scan_states["tau_s"].to_numpy(), -H_ALPHA)
        fv = np.power(1.0 + scan_states["tau_v"].to_numpy(),  H_BETA)
        fc = np.power(1.0 + scan_states["tau_c"].to_numpy(), -H_DELTA)
        # log-deviation from 1: |log10(factor)|.
        ds = np.abs(np.log10(np.clip(fs, 1e-30, None)))
        dv = np.abs(np.log10(np.clip(fv, 1e-30, None)))
        dc = np.abs(np.log10(np.clip(fc, 1e-30, None)))
        stacked = np.stack([ds, dv, dc], axis=1)
        winner = np.argmax(stacked, axis=1)
        all_neutral = (stacked.max(axis=1) < 1e-3)  # essentially uniform
        n_total = len(scan_states)
        labels = ["τ_s (freshness)", "τ_v (density)", "τ_c (crowding)"]
        print(f"Reconstructed {n_total} scan-time pheromone states (latest sample within stride).")
        print(f"  decisions effectively uniform (all factors ≈ 1):  {all_neutral.mean()*100:>5.1f}%")
        for i, lbl in enumerate(labels):
            sel = (winner == i) & (~all_neutral)
            mean_dev = stacked[sel, i].mean() if sel.any() else 0.0
            print(f"  decisions dominated by {lbl:<18s}  {sel.mean()*100:>5.1f}%   (mean |log10| dev = {mean_dev:.2f})")
    else:
        print("Could not reconstruct scan-time pheromone states.")
else:
    print("No scan events to analyse.")

# ============================================================
# 5. SCAN LOCALITY (consecutive scans per drone)
# ============================================================
banner("5. SCAN LOCALITY (where does each scan land relative to the last by the same drone?)")
if scan_df is not None and len(scan_df):
    # Build an undirected node-adjacency from graph_full so we can check "sibling edge".
    edge_nodes = graph_full[["from_node","to_node"]].astype(str)
    inc = {}  # node -> set(edge_index)
    for ei, row in edge_nodes.iterrows():
        for n in (row["from_node"], row["to_node"]):
            inc.setdefault(n, set()).add(ei)
    edge_neighbours = {}  # edge_index -> set(edge_index reachable via a shared node)
    for ei, row in edge_nodes.iterrows():
        nbrs = (inc.get(row["from_node"], set()) | inc.get(row["to_node"], set())) - {ei}
        edge_neighbours[ei] = nbrs

    sd = scan_df.sort_values(["drone_id","sim_time_s"]).copy()
    sd["prev_edge"] = sd.groupby("drone_id")["edge_index"].shift(1)
    sd = sd.dropna(subset=["prev_edge"]).astype({"prev_edge": int})
    n = len(sd)
    same = (sd["edge_index"] == sd["prev_edge"]).sum()
    is_sibling = sd.apply(lambda r: r["edge_index"] in edge_neighbours.get(int(r["prev_edge"]), set()), axis=1)
    sibling = is_sibling.sum() - same  # exclude the same-edge case
    further = n - same - sibling
    print(f"Of {n} consecutive scan pairs per drone:")
    print(f"  same edge      (looping in place):       {same:>6d}  ({same/n*100:>5.1f}%)")
    print(f"  sibling edge   (one-hop, normal local):  {sibling:>6d}  ({sibling/n*100:>5.1f}%)")
    print(f"  further away   (transit / off-route):    {further:>6d}  ({further/n*100:>5.1f}%)")
    print()
    print("Reading: a healthy ACO walk has most pairs as 'sibling' (one-hop). High 'same edge'")
    print("means drones are looping on a single edge; high 'further' means drones are flying")
    print("off-road between scans (TransitTarget jumps), which is fine but uses battery.")
else:
    print("No scans.")

# ============================================================
# 6. τ_v INFORMATIVENESS (does memory predict measurement?)
# ============================================================
banner("6. τ_v INFORMATIVENESS (pre-scan τ_v vs. measured density)")
if scan_df is not None and "tau_v" in (scan_states.columns if "scan_states" in dir() else []):
    # Use the reconstructed scan-time states; correlate pre-scan τ_v against measured density.
    enriched = scan_states.merge(scan_df[["sim_time_s","edge_index","measured_density_per_lane"]],
                                  on=["sim_time_s","edge_index"], how="left").dropna()
    if len(enriched) > 10:
        from scipy.stats import spearmanr
        rho, _ = spearmanr(enriched["tau_v"], enriched["measured_density_per_lane"])
        print(f"Spearman ρ between pre-scan τ_v and just-measured density:  {rho:>+.3f}  (over {len(enriched)} scans)")
        nz = enriched[enriched["tau_v"] > 0]
        if len(nz) > 5:
            rho_nz, _ = spearmanr(nz["tau_v"], nz["measured_density_per_lane"])
            print(f"   ... restricted to scans with prior τ_v > 0:           {rho_nz:>+.3f}  (over {len(nz)} scans)")
        print()
        print("Reading: ρ ≈ 0 means τ_v carries no info about what the drone is about to see.")
        print("ρ > 0.3 means the memory signal is predictive — re-visiting edges based on τ_v")
        print("is operationally justified. ρ < 0 would be a red flag (anti-predictive memory).")
else:
    print("Insufficient reconstructed states for correlation.")

# ============================================================
# 7. EDGE CONCENTRATION (how lopsided is scan distribution?)
# ============================================================
banner("7. EDGE CONCENTRATION OF SCANS")
if scan_df is not None and len(scan_df):
    counts = scan_df.groupby("edge_index").size().sort_values(ascending=False)
    cum = counts.cumsum() / counts.sum() * 100
    # What % of scans landed on the top X% of edges?
    for top_pct in [1, 5, 10, 25]:
        cutoff_idx = max(1, int(np.ceil(len(counts) * top_pct / 100.0)))
        share = counts.iloc[:cutoff_idx].sum() / counts.sum() * 100
        print(f"Top {top_pct:>2d}% of scanned edges ({cutoff_idx:>4d}) absorb  {share:>5.1f}%  of all scans")
    # Gini coefficient (on scanned edges only, since unscanned ones trivially get 0).
    sorted_c = np.sort(counts.values.astype(float))
    n_c = len(sorted_c)
    gini = (2 * np.sum((np.arange(1, n_c + 1)) * sorted_c) - (n_c + 1) * sorted_c.sum()) / (n_c * sorted_c.sum())
    print(f"\nGini coefficient over scanned edges: {gini:.3f}   (0=uniform, 1=one edge gets all scans)")
    print()
    print("Reading: Gini < 0.3 = even coverage. 0.3-0.5 = moderate concentration (some priority edges).")
    print(">0.5 = strongly lopsided. >0.7 = lock-in — a handful of edges absorb most attention.")

# ============================================================
# 8. VERDICT (heuristic flags)
# ============================================================
banner("8. VERDICT — heuristic flags")
flags = []
if "stacked" in dir():
    # β-effectiveness check
    if (stacked[:, 1] < 1e-3).mean() > 0.95:
        flags.append("β factor is ≈1 in >95% of scan moments → β is effectively unused. Lower the threshold or check Qv/RhoV.")
    if (stacked[:, 0] < 1e-3).mean() > 0.95:
        flags.append("α factor is ≈1 in >95% of scan moments → α is effectively unused.")
    if (stacked[:, 2] < 1e-3).mean() > 0.95:
        flags.append("δ factor is ≈1 in >95% of scan moments → δ is effectively unused.")
if "gini" in dir() and gini > 0.7:
    flags.append(f"Lock-in (Gini={gini:.2f}>0.7): swarm scanning the same edges repeatedly. Likely τ_v runaway.")
if "edges_ever_scanned" in dir() and n_edges > 0 and edges_ever_scanned / n_edges < 0.20:
    flags.append(f"Low coverage ({edges_ever_scanned}/{n_edges}={edges_ever_scanned/n_edges*100:.1f}%): swarm not exploring the network in this time window.")
if "same" in dir() and "n" in dir() and n > 0 and same/n > 0.25:
    flags.append(f"Looping detected: {same/n*100:.1f}% of consecutive scan pairs are on the SAME edge. Drones stuck.")
if not flags:
    flags.append("No red flags from heuristics. Look at panels 1-7 for fine-grained tuning hints.")
for f in flags:
    print(f"  • {f}")
print()
